# Handwritten Digit Recognition with CNN (MNIST)


## Updates
- Slide CNN (16→32 filters, Dense 64) as the core model
- Size-matched ANN baseline
- Wider CNN (two convolutions per block)
- BatchNorm + Dropout
- Data augmentation
- 3-model ensemble (E6)
- Convolution and pooling demo
- Learned filters and feature maps
- Tuning sweep with noise estimate
- Stratified 5-fold cross-validation
- Confusion pairs, confidence and coverage analysis
- Save, reload, predict
- Quick/full run profile


## 1. Setup

In [ ]:
import os

# keep TensorFlow's C++ start-up chatter out of the report (must be set before the import)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import json
import platform
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split

# ----------------------------------------------------------------------------- reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

# ----------------------------------------------------------------------------- run profile
# "full"  -> the configuration used for the numbers in the report
# "quick" -> a ~2 minute smoke test that exercises every cell (for debugging)
PROFILE = "full"

if PROFILE == "full":
    CFG = dict(
        ep_ann=15,       # E1  fully connected baseline
        ep_cnn=15,       # E2  CNN from the slides, E3 wider CNN
        ep_bn=18,        # E4  + BatchNorm + Dropout
        ep_aug=18,       # E5  + data augmentation (a harder task, so it needs the full budget)
        ep_tune=5,       # short screening runs for the hyperparameter sweeps
        n_ensemble=3,    # E6  number of CNNs in the committee
        cv_folds=5,      # stratified k-fold
        cv_epochs=4,
        train_subset=None,
    )
else:
    CFG = dict(ep_ann=2, ep_cnn=2, ep_bn=2, ep_aug=3, ep_tune=1,
               n_ensemble=2, cv_folds=3, cv_epochs=2, train_subset=10000)

# ----------------------------------------------------------------------------- plot style
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.titleweight": "bold",
    "legend.frameon": False,
})
# colour-blind-safe qualitative palette, used consistently for every experiment
PALETTE = ["#4C6EF5", "#F76707", "#0CA678", "#AE3EC9", "#F59F00", "#495057"]
COLOR = {}  # experiment name -> colour, filled in as experiments are registered

def color_for(name):
    """Give every experiment a stable colour so all figures agree with each other."""
    if name not in COLOR:
        COLOR[name] = PALETTE[len(COLOR) % len(PALETTE)]
    return COLOR[name]

print("Python          :", platform.python_version())
print("TensorFlow      :", tf.__version__)
print("Keras           :", keras.__version__)
print("Platform        :", platform.platform())
gpus = tf.config.list_physical_devices("GPU")
print("GPU devices     :", gpus if gpus else "none (running on CPU)")
print("Profile         :", PROFILE)
print("Configuration   :", json.dumps(CFG))

## 2. Problem Case and Data

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Training images :", x_train_full.shape, x_train_full.dtype)
print("Training labels :", y_train_full.shape, y_train_full.dtype)
print("Test images     :", x_test.shape, x_test.dtype)
print("Test labels     :", y_test.shape)
print("Pixel range     :", x_train_full.min(), "to", x_train_full.max())
print("Classes         :", np.unique(y_train_full))

dist = pd.DataFrame({
    "digit": np.arange(10),
    "train_count": np.bincount(y_train_full),
    "test_count": np.bincount(y_test),
})
dist["train_share_%"] = (100 * dist["train_count"] / len(y_train_full)).round(2)
print()
print("Class distribution")
print(dist.to_string(index=False))
print()
print(f"Most common class is {dist['train_count'].max() / dist['train_count'].min():.2f}x "
      f"the rarest -> the dataset is close to balanced, so plain accuracy is a fair metric "
      f"and no class weighting is needed.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

# (a) class distribution --------------------------------------------------------------------
ax = axes[0, 0]
ax.bar(dist["digit"], dist["train_count"], color=PALETTE[0], label="train")
ax.bar(dist["digit"], dist["test_count"], color=PALETTE[1], label="test")
ax.set_xticks(range(10))
ax.set_title("(a) Class distribution is nearly balanced")
ax.set_xlabel("digit")
ax.set_ylabel("number of images")
ax.legend()

# (b) sample images -------------------------------------------------------------------------
ax = axes[0, 1]
rng = np.random.default_rng(SEED)
grid_rows = []
for d in range(10):                      # one row per digit, 8 random examples each
    idx = rng.choice(np.where(y_train_full == d)[0], 8, replace=False)
    grid_rows.append(np.concatenate(list(x_train_full[idx]), axis=1))
ax.imshow(np.concatenate(grid_rows, axis=0), cmap="gray")
ax.set_title("(b) Eight random examples of each digit")
ax.set_ylabel("digit 0 (top) to 9 (bottom)")
ax.set_xticks([])
ax.set_yticks([])
ax.grid(False)

# (c) average image per class ---------------------------------------------------------------
ax = axes[1, 0]
means = [x_train_full[y_train_full == d].mean(axis=0) for d in range(10)]
ax.imshow(np.concatenate([np.concatenate(means[:5], axis=1),
                          np.concatenate(means[5:], axis=1)], axis=0), cmap="magma")
ax.set_title("(c) Average image per class: digits are centred,\nbut the ink is blurred -> high within-class variation")
ax.set_xticks([])
ax.set_yticks([])
ax.grid(False)

# (d) pixel intensity histogram -------------------------------------------------------------
ax = axes[1, 1]
ax.hist(x_train_full.ravel(), bins=50, color=PALETTE[2])
ax.set_yscale("log")
ax.set_title("(d) Pixel intensities: mostly empty black background")
ax.set_xlabel("pixel value (0-255)")
ax.set_ylabel("count (log scale)")
ink = (x_train_full > 0).mean()
ax.annotate(f"only {100 * ink:.1f}% of pixels\ncarry any ink",
            xy=(0.45, 0.6), xycoords="axes fraction", fontsize=9)

plt.tight_layout()
plt.show()

## 3. Preprocessing and Splitting

In [ ]:
# 1 + 2: scale to [0, 1] and add the single grayscale channel
x_train_full = (x_train_full.astype("float32") / 255.0)[..., np.newaxis]
x_test = (x_test.astype("float32") / 255.0)[..., np.newaxis]

if CFG["train_subset"]:                  # only used by PROFILE = "quick"
    x_train_full = x_train_full[: CFG["train_subset"]]
    y_train_full = y_train_full[: CFG["train_subset"]]

# stratified 90 / 10 split of the training file
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=0.1, stratify=y_train_full, random_state=SEED,
)

INPUT_SHAPE = x_train.shape[1:]
N_CLASSES = 10

print(f"train      : {x_train.shape}  labels {y_train.shape}")
print(f"validation : {x_val.shape}  labels {y_val.shape}")
print(f"test       : {x_test.shape}  labels {y_test.shape}")
print(f"input shape to the network : {INPUT_SHAPE}")
print(f"pixel range after scaling  : {x_train.min():.1f} to {x_train.max():.1f}")
print()

check = pd.DataFrame({
    "digit": range(10),
    "train_%": (100 * np.bincount(y_train, minlength=10) / len(y_train)).round(2),
    "val_%": (100 * np.bincount(y_val, minlength=10) / len(y_val)).round(2),
})
print(check.to_string(index=False))

# 3: the two loss functions are the same thing, written differently [slides p. 26]
probs = np.array([[0.02, 0.90, 0.08]], dtype="float32")
sparse = keras.losses.sparse_categorical_crossentropy(np.array([1]), probs).numpy()[0]
onehot = keras.losses.categorical_crossentropy(np.array([[0.0, 1.0, 0.0]]), probs).numpy()[0]
print()
print(f"sparse_categorical_crossentropy = {sparse:.6f}")
print(f"categorical_crossentropy        = {onehot:.6f}   <- identical, as expected")

## 4. Convolution and Pooling

In [ ]:
def conv2d_once(image, kernel):
    """Convolve one 28x28 image with one 3x3 kernel, exactly as a Conv2D layer would."""
    x = tf.constant(image.reshape(1, 28, 28, 1), dtype=tf.float32)
    k = tf.constant(np.asarray(kernel, dtype="float32").reshape(3, 3, 1, 1))
    return tf.nn.conv2d(x, k, strides=1, padding="SAME").numpy()[0, :, :, 0]

# the kernel printed on slide 16, plus two other classical hand-crafted filters
KERNELS = {
    "edge detector\n(the slide's kernel)": [[0, -1, 0], [-1, 4, -1], [0, -1, 0]],
    "vertical edges\n(Sobel x)": [[1, 0, -1], [2, 0, -2], [1, 0, -1]],
    "horizontal edges\n(Sobel y)": [[1, 2, 1], [0, 0, 0], [-1, -2, -1]],
}

sample = x_train[np.where(y_train == 3)[0][0]].squeeze()

fig, axes = plt.subplots(2, 4, figsize=(12, 5.6))

axes[0, 0].imshow(sample, cmap="gray")
axes[0, 0].set_title("input image\n28 x 28 x 1")
axes[1, 0].axis("off")

for col, (name, kernel) in enumerate(KERNELS.items(), start=1):
    feature_map = conv2d_once(sample, kernel)
    axes[0, col].imshow(feature_map, cmap="gray")
    axes[0, col].set_title(f"feature map after\n{name}")
    # draw the 3x3 kernel itself underneath, with its nine numbers
    k = np.asarray(kernel, dtype="float32")
    axes[1, col].imshow(k, cmap="coolwarm", vmin=-4, vmax=4)
    for i in range(3):
        for j in range(3):
            axes[1, col].text(j, i, f"{k[i, j]:.0f}", ha="center", va="center", fontsize=11)
    axes[1, col].set_title("the 3 x 3 kernel\n(9 weights)", fontsize=9)

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)

fig.suptitle("A 3 x 3 convolution extracts an interpretable feature.  In a CNN these nine "
             "numbers are learned, not chosen by hand  [slides p. 16-17, 19-20]",
             fontsize=10, y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Max pooling, and the shape arithmetic of the architecture from the slides [slides p. 21-24]
pooled_1 = tf.nn.max_pool2d(sample.reshape(1, 28, 28, 1), ksize=2, strides=2, padding="SAME")
pooled_2 = tf.nn.max_pool2d(pooled_1, ksize=2, strides=2, padding="SAME")

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, img, title in zip(
    axes,
    [sample, pooled_1.numpy()[0, :, :, 0], pooled_2.numpy()[0, :, :, 0]],
    ["original  28 x 28", "after 2x2 max pool  14 x 14", "after a second 2x2 max pool  7 x 7"],
):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
fig.suptitle("Pooling halves each side and keeps the strongest response - the digit stays "
             "recognisable with 1/16 of the pixels  [slides p. 21]", fontsize=10)
plt.tight_layout()
plt.show()

shapes = pd.DataFrame([
    ("input",                       "28 x 28 x 1",   0,      "grayscale image"),
    ("Conv2D 16 filters 3x3, same", "28 x 28 x 16",  160,    "16 learned feature maps; 'same' padding keeps 28x28"),
    ("MaxPooling2D 2x2",            "14 x 14 x 16",  0,      "downsample, no weights"),
    ("Conv2D 32 filters 3x3, same", "14 x 14 x 32",  4640,   "combines the 16 maps into 32 richer ones"),
    ("MaxPooling2D 2x2",            "7 x 7 x 32",    0,      "downsample again"),
    ("Flatten",                     "1568",          0,      "7 x 7 x 32 = 1568, matrix becomes a vector"),
    ("Dense 64, relu",              "64",            100416, "fully connected hidden layer"),
    ("Dense 10, softmax",           "10",            650,    "one neuron per digit, probabilities sum to 1"),
], columns=["layer", "output shape", "parameters", "what it does"])

print(shapes.to_string(index=False))
print(f"\nTotal parameters: {shapes['parameters'].sum():,}")

## 5. Training Helpers

In [ ]:
results = []       # the main ladder E1..E6
tuning = []        # the hyperparameter sweeps of section 8
histories = {}     # experiment name -> keras History.history
models = {}        # experiment name -> trained keras model
predictors = {}    # experiment name -> callable(x) -> (n, 10) probability matrix

def make_optimizer(name, lr):
    """Build one of the three optimisers we compare in section 8."""
    name = name.lower()
    if name == "sgd":
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=lr)
    if name == "rmsprop":
        return keras.optimizers.RMSprop(learning_rate=lr)
    raise ValueError(f"Unknown optimizer: {name!r}")

class EpochLog(keras.callbacks.Callback):
    """One compact line per epoch - easier to read in a report than the default progress bar."""

    def __init__(self, every=1):
        super().__init__()
        self.every = every

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if epoch == 0 or (epoch + 1) % self.every == 0:
            try:
                lr = float(keras.ops.convert_to_numpy(self.model.optimizer.learning_rate))
            except Exception:
                lr = float("nan")
            print(f"    epoch {epoch + 1:>3}  loss {logs.get('loss', 0):.4f}  "
                  f"acc {logs.get('accuracy', 0):.4f}  |  "
                  f"val_loss {logs.get('val_loss', 0):.4f}  "
                  f"val_acc {logs.get('val_accuracy', 0):.4f}  |  lr {lr:.1e}")

def run_experiment(name, model, optimizer="adam", lr=1e-3, epochs=15, batch_size=128,
                   patience=5, store=None, keep=True, log=True, note=""):
    """Compile, train and evaluate one model, and append one row to a results list.

    The same routine is used for every experiment so that the comparison is fair.
    """
    store = results if store is None else store
    model.compile(
        optimizer=make_optimizer(optimizer, lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    callbacks = [
        EarlyStopping(monitor="val_accuracy", mode="max",
                      patience=patience, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=max(2, patience - 2), min_lr=1e-5, verbose=0),
    ]
    if log:
        callbacks.append(EpochLog())
        print(f"[{name}] {note or model.name}  "
              f"({model.count_params():,} parameters, up to {epochs} epochs)")

    t0 = time.perf_counter()
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs, batch_size=batch_size,
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.perf_counter() - t0

    val_acc = model.evaluate(x_val, y_val, verbose=0)[1]
    test_acc = model.evaluate(x_test, y_test, verbose=0)[1]

    store.append({
        "experiment": name,
        "model": model.name,
        "optimizer": optimizer,
        "lr": lr,
        "batch_size": batch_size,
        "epochs_run": len(history.history["loss"]),
        "params": model.count_params(),
        "val_acc": round(float(val_acc), 4),
        "test_acc": round(float(test_acc), 4),
        "test_err_%": round(100 * (1 - float(test_acc)), 3),
        "time_s": round(elapsed, 1),
        "note": note,
    })
    histories[name] = history.history
    if keep:
        models[name] = model
        predictors[name] = lambda x, _m=model: _m.predict(x, verbose=0)
    color_for(name)

    if log:
        print(f"  -> val_acc {val_acc:.4f} | test_acc {test_acc:.4f} "
              f"| test error {100 * (1 - test_acc):.2f}% "
              f"| {len(history.history['loss'])} epochs in {elapsed:.0f}s\n")
    return model

def plot_history(names, title=None):
    """Accuracy and loss curves, training (solid) vs validation (dashed)."""
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
    for n in names:
        h, c = histories[n], color_for(n)
        ep = range(1, len(h["loss"]) + 1)
        axes[0].plot(ep, h["accuracy"], color=c, label=f"{n} (train)")
        axes[0].plot(ep, h["val_accuracy"], color=c, ls="--", label=f"{n} (val)")
        axes[1].plot(ep, h["loss"], color=c, label=f"{n} (train)")
        axes[1].plot(ep, h["val_loss"], color=c, ls="--", label=f"{n} (val)")
    axes[0].set_title("Accuracy")
    axes[0].set_ylabel("accuracy")
    axes[1].set_title("Loss (cross-entropy)")
    axes[1].set_ylabel("loss")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.legend(fontsize=7.5)
    if title:
        fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()

def gap_report(name):
    """Report the train-validation gap, i.e. how much the model overfits."""
    h = histories[name]
    best = int(np.argmax(h["val_accuracy"]))
    gap = h["accuracy"][best] - h["val_accuracy"][best]
    print(f"{name:<22} best epoch {best + 1:>2}  "
          f"train {h['accuracy'][best]:.4f}  val {h['val_accuracy'][best]:.4f}  "
          f"overfitting gap {gap:+.4f}")
    return gap

# reserve one distinct colour for each rung of the ladder, before the sweeps consume any
for _name in ["E1_ann", "E2_cnn_slide", "E3_cnn_wide", "E4_cnn_bn", "E5_cnn_bn_aug",
              "E6_ensemble"]:
    color_for(_name)

print("Helpers ready.")

## 6. Model Architectures

In [ ]:
def ann_baseline():
    """M1: fully connected network, the control for 'what does convolution buy us?'"""
    return keras.Sequential([
        layers.Input(INPUT_SHAPE),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(N_CLASSES, activation="softmax"),
    ], name="ann_baseline")

def cnn_slide():
    """M2: the CNN from the course slides, reproduced exactly [slides p. 23-24]."""
    return keras.Sequential([
        layers.Input(INPUT_SHAPE),
        layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(N_CLASSES, activation="softmax"),
    ], name="cnn_slide")

def cnn_wide():
    """M3: wider, two 3x3 convolutions per block (the VGG principle [slides p. 30])."""
    return keras.Sequential([
        layers.Input(INPUT_SHAPE),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(N_CLASSES, activation="softmax"),
    ], name="cnn_wide")

def augmentation_layers():
    """Random rotation / shift / zoom, applied only while training.

    Factors are deliberately small. In Keras, RandomRotation's factor is a fraction of a full
    turn, so 0.03 is about +/- 11 degrees - enough to cover normal handwriting slant without
    turning a 6 into a 9. There is no horizontal flip: a mirrored digit is not a digit.
    Empty space is filled with black (fill_value=0) to match the MNIST background.
    """
    return [
        layers.RandomRotation(0.03, fill_mode="constant", fill_value=0.0),
        layers.RandomTranslation(0.10, 0.10, fill_mode="constant", fill_value=0.0),
        layers.RandomZoom(0.10, fill_mode="constant", fill_value=0.0),
    ]

# BatchNorm keeps a running mean/variance that it uses at prediction time. Keras defaults to
# momentum 0.99, which needs a few thousand updates to settle; with a short epoch budget the
# running statistics would still be near their initial values and validation accuracy would
# collapse even though training accuracy looks fine. 0.9 settles in a few hundred updates, so
# the model behaves correctly in both the "quick" and the "full" profile.
BN_MOMENTUM = 0.9

def cnn_bn(dropout_conv=0.25, dropout_head=0.5, augment=False, name=None):
    """M4: cnn_wide + BatchNorm + Dropout, optionally with data augmentation in front."""
    stack = [layers.Input(INPUT_SHAPE)]
    if augment:
        stack += augmentation_layers()

    for filters in (32, 64):                       # two convolutional blocks
        for _ in range(2):                         # two convolutions inside each block
            stack += [
                layers.Conv2D(filters, (3, 3), padding="same", use_bias=False),
                layers.BatchNormalization(momentum=BN_MOMENTUM),
                layers.Activation("relu"),
            ]
        stack += [layers.MaxPooling2D((2, 2)), layers.Dropout(dropout_conv)]

    stack += [
        layers.Flatten(),
        layers.Dense(128, use_bias=False),
        layers.BatchNormalization(momentum=BN_MOMENTUM),
        layers.Activation("relu"),
        layers.Dropout(dropout_head),
        layers.Dense(N_CLASSES, activation="softmax"),
    ]
    return keras.Sequential(stack, name=name or ("cnn_bn_aug" if augment else "cnn_bn"))

# the architecture from the slides, printed with model.summary() as the material does [p. 24]
cnn_slide().summary()

In [ ]:
overview = []
for builder, label in [(ann_baseline, "M1 fully connected (no convolution)"),
                       (cnn_slide, "M2 CNN from the slides [p. 23-24]"),
                       (cnn_wide, "M3 wider CNN, 2 conv per block"),
                       (cnn_bn, "M4 M3 + BatchNorm + Dropout")]:
    m = builder()
    overview.append({
        "architecture": label,
        "conv layers": sum(isinstance(l, layers.Conv2D) for l in m.layers),
        "pool layers": sum(isinstance(l, layers.MaxPooling2D) for l in m.layers),
        "dense layers": sum(isinstance(l, layers.Dense) for l in m.layers),
        "parameters": f"{m.count_params():,}",
    })
    del m

print(pd.DataFrame(overview).to_string(index=False))

## 7. Experiments E1-E5

In [ ]:
# ---------------------------------------------------------------- E1: no convolution (control)
run_experiment("E1_ann", ann_baseline(), epochs=CFG["ep_ann"],
               note="fully connected baseline, no convolution")

# ---------------------------------------------------------------- E2: the CNN from the slides
run_experiment("E2_cnn_slide", cnn_slide(), epochs=CFG["ep_cnn"],
               note="CNN from the course slides [p. 23-24]")

plot_history(["E1_ann", "E2_cnn_slide"],
             "E1 vs E2 - the effect of convolution")

e1 = results[0]
e2 = results[1]
pd.DataFrame([e1, e2])[["experiment", "params", "val_acc", "test_acc", "test_err_%"]]

In [ ]:
# The loss comparison the material draws on slide 28, reproduced for our ANN and CNN.
fig, ax = plt.subplots(figsize=(7, 4))
for name, label in [("E1_ann", "ANN"), ("E2_cnn_slide", "CNN")]:
    h, c = histories[name], color_for(name)
    ep = range(1, len(h["loss"]) + 1)
    ax.plot(ep, h["loss"], color=c, marker="o", ms=3, label=f"train loss {label}")
    ax.plot(ep, h["val_loss"], color=c, ls="--", marker="+", ms=6, label=f"val loss {label}")
ax.set_xlabel("epoch")
ax.set_ylabel("cross-entropy loss")
ax.set_title("Loss comparison of the CNN and the ANN model  [reproduces slides p. 28]")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------- E3: more capacity (wider, 2 conv per block)
run_experiment("E3_cnn_wide", cnn_wide(), epochs=CFG["ep_cnn"],
               note="wider CNN, two 3x3 convolutions per block")

# ------------------------------------------------- E4: regularisation (BatchNorm + Dropout)
run_experiment("E4_cnn_bn", cnn_bn(), epochs=CFG["ep_bn"],
               note="E3 + BatchNormalization + Dropout")

plot_history(["E2_cnn_slide", "E3_cnn_wide", "E4_cnn_bn"],
             "E2 -> E3 -> E4 - more capacity, then regularisation")

for n in ["E2_cnn_slide", "E3_cnn_wide", "E4_cnn_bn"]:
    gap_report(n)

In [ ]:
# What the augmentation layers actually do to a digit, before we train with them.
aug = keras.Sequential(augmentation_layers(), name="augmenter")
demo = x_train[np.where(y_train == 5)[0][0]][np.newaxis, ...]

fig, axes = plt.subplots(2, 7, figsize=(12, 3.8))
axes[0, 0].imshow(demo[0].squeeze(), cmap="gray")
axes[0, 0].set_title("original", fontsize=9)
axes[1, 0].axis("off")
for i, ax in enumerate([*axes[0, 1:], *axes[1, 1:]]):
    ax.imshow(aug(demo, training=True)[0].numpy().squeeze(), cmap="gray")
    ax.set_title(f"augmented {i + 1}", fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
fig.suptitle("Data augmentation: one training image becomes an unlimited supply of plausible "
             "variants\n(small rotation, shift and zoom - never a flip)", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------- E5: the same model, trained with augmentation
run_experiment("E5_cnn_bn_aug", cnn_bn(augment=True), epochs=CFG["ep_aug"], patience=7,
               note="E4 + random rotation / shift / zoom")

plot_history(["E4_cnn_bn", "E5_cnn_bn_aug"], "E4 vs E5 - the effect of data augmentation")

e4 = next(r for r in results if r["experiment"] == "E4_cnn_bn")
e5 = next(r for r in results if r["experiment"] == "E5_cnn_bn_aug")
print(f"E4 test error : {e4['test_err_%']:.2f}%")
print(f"E5 test error : {e5['test_err_%']:.2f}%")
print()
for n in ["E4_cnn_bn", "E5_cnn_bn_aug"]:
    gap_report(n)

## 8. Hyperparameter Tuning

In [ ]:
EP = CFG["ep_tune"]

for opt, lr in [("sgd", 1e-2), ("adam", 1e-3), ("rmsprop", 1e-3)]:
    run_experiment(f"T_opt_{opt}", cnn_slide(), optimizer=opt, lr=lr,
                   epochs=EP, patience=EP, store=tuning, keep=False, log=False)
    print(f"  {opt:<8} lr={lr:<7} val_acc {tuning[-1]['val_acc']:.4f}")

for lr in [1e-2, 1e-3, 1e-4]:
    run_experiment(f"T_lr_{lr:g}", cnn_slide(), lr=lr,
                   epochs=EP, patience=EP, store=tuning, keep=False, log=False)
    print(f"  lr={lr:<9g} val_acc {tuning[-1]['val_acc']:.4f}")

for bs in [32, 128, 512]:
    run_experiment(f"T_bs_{bs}", cnn_slide(), batch_size=bs,
                   epochs=EP, patience=EP, store=tuning, keep=False, log=False)
    print(f"  batch={bs:<7} val_acc {tuning[-1]['val_acc']:.4f}  "
          f"({tuning[-1]['time_s']:.0f}s)")

for dr in [0.0, 0.25, 0.5]:
    run_experiment(f"T_dropout_{dr}", cnn_bn(dropout_conv=dr, dropout_head=min(2 * dr, 0.6)),
                   epochs=EP, patience=EP, store=tuning, keep=False, log=False)
    print(f"  dropout={dr:<5} val_acc {tuning[-1]['val_acc']:.4f}")

tuning_df = pd.DataFrame(tuning)[["experiment", "optimizer", "lr", "batch_size",
                                  "val_acc", "test_acc", "time_s"]]
tuning_df

In [ ]:
groups = [("T_opt", "Optimiser"), ("T_lr", "Learning rate"),
          ("T_bs", "Batch size"), ("T_dropout", "Dropout rate")]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6), sharey=True)
for ax, (prefix, title) in zip(axes, groups):
    sub = tuning_df[tuning_df["experiment"].str.startswith(prefix)]
    labels = sub["experiment"].str.replace(prefix + "_", "", regex=False)
    best = sub["val_acc"].idxmax()
    colors = [PALETTE[2] if i == best else PALETTE[0] for i in sub.index]
    bars = ax.bar(labels, sub["val_acc"], color=colors)
    for b, v in zip(bars, sub["val_acc"]):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.4f}",
                ha="center", va="bottom", fontsize=7.5, rotation=90)
    ax.set_title(title)
    ax.set_ylim(max(0, tuning_df["val_acc"].min() - 0.02), 1.015)
axes[0].set_ylabel("validation accuracy")
fig.suptitle(f"Hyperparameter sweeps ({EP}-epoch screening runs) - green is the winner "
             f"of each group", fontsize=10)
plt.tight_layout()
plt.show()

# How much does each hyperparameter actually matter? -> the spread it produces.
sweep_spreads = {}
for prefix, title in groups:
    sub = tuning_df[tuning_df["experiment"].str.startswith(prefix)]
    row = sub.loc[sub["val_acc"].idxmax()]
    spread = sub["val_acc"].max() - sub["val_acc"].min()
    sweep_spreads[title] = spread
    print(f"  {title:<15} best = {row['experiment'].replace(prefix + '_', ''):<8} "
          f"val_acc {row['val_acc']:.4f}   |  spread across its values: {spread:.4f}")

most_sensitive = max(sweep_spreads, key=sweep_spreads.get)
most_sensitive_spread = sweep_spreads[most_sensitive]

# A free noise estimate: two of the runs above are the SAME configuration, trained twice.
dup = tuning_df[tuning_df["experiment"].isin(["T_opt_adam", "T_lr_0.001"])]["val_acc"].tolist()
noise = abs(dup[0] - dup[1]) if len(dup) == 2 else float("nan")

print(f"\nMost sensitive: {most_sensitive} (spread {most_sensitive_spread:.4f}) | "
      f"noise floor (same config twice): {noise:.4f}")

## 9. E6 Ensemble

In [ ]:
ensemble_models = [models["E5_cnn_bn_aug"]]      # member 1 is the E5 model, already trained
ensemble_time = next(r for r in results if r["experiment"] == "E5_cnn_bn_aug")["time_s"]

for i in range(2, CFG["n_ensemble"] + 1):
    keras.utils.set_random_seed(SEED + 100 * i)   # a different seed for every member
    name = f"E6_member{i}"
    m = run_experiment(name, cnn_bn(augment=True, name=f"cnn_bn_aug_s{i}"),
                       epochs=CFG["ep_aug"], patience=7, store=tuning, keep=False, log=True,
                       note=f"ensemble member {i} (seed {SEED + 100 * i})")
    ensemble_models.append(m)
    ensemble_time += tuning[-1]["time_s"]

keras.utils.set_random_seed(SEED)                 # restore the global seed

def ensemble_predict(x, members=None):
    """Average the softmax outputs of every member - a soft-voting ensemble."""
    members = members or ensemble_models
    return np.mean([m.predict(x, verbose=0) for m in members], axis=0)

ens_val = ensemble_predict(x_val)
ens_test = ensemble_predict(x_test)
ens_val_acc = float((ens_val.argmax(1) == y_val).mean())
ens_test_acc = float((ens_test.argmax(1) == y_test).mean())

# the epochs actually used by the members (early stopping may have cut them short)
member_names = ["E5_cnn_bn_aug"] + [f"E6_member{i}" for i in range(2, CFG["n_ensemble"] + 1)]
member_epochs = sum(len(histories[n]["loss"]) for n in member_names if n in histories)

results.append({
    "experiment": "E6_ensemble", "model": f"{len(ensemble_models)} x cnn_bn_aug",
    "optimizer": "adam", "lr": 1e-3, "batch_size": 128,
    "epochs_run": member_epochs,
    "params": sum(m.count_params() for m in ensemble_models),
    "val_acc": round(ens_val_acc, 4), "test_acc": round(ens_test_acc, 4),
    "test_err_%": round(100 * (1 - ens_test_acc), 3),
    "time_s": round(ensemble_time, 1),
    "note": f"soft-voting average of {len(ensemble_models)} augmented CNNs",
})
predictors["E6_ensemble"] = ensemble_predict
color_for("E6_ensemble")

print(f"\nIndividual members on the test set:")
for i, m in enumerate(ensemble_models, 1):
    acc = float((m.predict(x_test, verbose=0).argmax(1) == y_test).mean())
    print(f"  member {i} ({m.name:<14}) test_acc {acc:.4f}   error {100 * (1 - acc):.2f}%")
print(f"\n  ENSEMBLE of {len(ensemble_models)}            test_acc {ens_test_acc:.4f}   "
      f"error {100 * (1 - ens_test_acc):.2f}%")

## 10. Results Summary

In [ ]:
summary = (pd.DataFrame(results)
           .sort_values("val_acc", ascending=False)
           .reset_index(drop=True))

display_cols = ["experiment", "model", "epochs_run", "params",
                "val_acc", "test_acc", "test_err_%", "time_s", "note"]
print(summary[display_cols].to_string(index=False))

best_name = summary.loc[0, "experiment"]
baseline = next(r for r in results if r["experiment"] == "E1_ann")
print()
print(f"Selected best model : {best_name}")
print(f"  validation accuracy {summary.loc[0, 'val_acc']:.4f}")
print(f"  test accuracy       {summary.loc[0, 'test_acc']:.4f}  "
      f"(error {summary.loc[0, 'test_err_%']:.2f}%)")
print(f"  vs the E1 baseline  {baseline['test_acc']:.4f}  "
      f"(error {baseline['test_err_%']:.2f}%)")
print(f"  -> {(1 - summary.loc[0, 'test_err_%'] / baseline['test_err_%']) * 100:.1f}% of the "
      f"baseline's errors removed; "
      f"{baseline['test_err_%'] / summary.loc[0, 'test_err_%']:.1f}x fewer mistakes.")

In [ ]:
ladder = [r for r in results if r["experiment"].startswith(("E1", "E2", "E3", "E4", "E5", "E6"))]
ladder_df = pd.DataFrame(ladder)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) test error rate per rung of the ladder -------------------------------------------------
ax = axes[0]
names = ladder_df["experiment"]
bars = ax.bar(range(len(names)), ladder_df["test_err_%"],
              color=[color_for(n) for n in names])
for b, v in zip(bars, ladder_df["test_err_%"]):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}%", ha="center", va="bottom", fontsize=8)
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.replace("_", "\n", 1) for n in names], fontsize=7.5)
ax.set_ylabel("test error rate (%)   lower is better")
ax.set_title("(a) Every rung of the ladder removes errors")

# (b) accuracy, zoomed in where the action is ------------------------------------------------
ax = axes[1]
ax.plot(range(len(names)), ladder_df["val_acc"], "o-", color=PALETTE[0], label="validation")
ax.plot(range(len(names)), ladder_df["test_acc"], "s--", color=PALETTE[1], label="test")
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.split("_")[0] for n in names])
ax.set_ylim(min(ladder_df["val_acc"].min(), ladder_df["test_acc"].min()) - 0.002, 1.0005)
ax.set_ylabel("accuracy")
ax.set_title("(b) Validation and test agree\n(no overfitting to the validation set)")
ax.legend()

# (c) accuracy vs model size -----------------------------------------------------------------
ax = axes[2]
for _, r in ladder_df.iterrows():
    ax.scatter(r["params"], r["test_err_%"], s=90, color=color_for(r["experiment"]), zorder=3)
    ax.annotate(r["experiment"].split("_")[0],
                (r["params"], r["test_err_%"]), textcoords="offset points",
                xytext=(7, 3), fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("number of parameters (log scale)")
ax.set_ylabel("test error rate (%)")
ax.set_title("(c) Accuracy is not just about size:\nE2 beats E1 at the same parameter count")

plt.tight_layout()
plt.show()

## 11. Final Evaluation

In [ ]:
best_name = summary.loc[0, "experiment"]
best_predict = predictors[best_name]

y_prob = best_predict(x_test)
y_pred = y_prob.argmax(axis=1)
y_conf = y_prob.max(axis=1)

print(f"Final evaluation of: {best_name}\n")
print(classification_report(y_test, y_pred, digits=4,
                            target_names=[f"digit {d}" for d in range(10)]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))

for ax, matrix, fmt, title, cmap in [
    (axes[0], cm, "d", "(a) Confusion matrix - counts", "Blues"),
    (axes[1], 100 * cm_norm, ".1f", "(b) Row-normalised (% of each true class)", "Blues"),
]:
    im = ax.imshow(matrix, cmap=cmap, vmax=np.max(matrix[~np.eye(10, dtype=bool)]) * 2.2)
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))
    ax.set_xlabel("predicted digit")
    ax.set_ylabel("true digit")
    ax.set_title(title)
    ax.grid(False)
    for i in range(10):
        for j in range(10):
            v = matrix[i, j]
            if i == j:
                txt = f"{v:{fmt}}"
                ax.text(j, i, txt, ha="center", va="center", fontsize=7.5,
                        color="#0b3d91", fontweight="bold")
            elif v > 0:
                ax.text(j, i, f"{v:{fmt}}", ha="center", va="center", fontsize=7.5, color="#c92a2a")

fig.suptitle(f"Confusion matrix of {best_name} on the 10,000 test images "
             f"(diagonal = correct, off-diagonal = mistakes)", fontsize=10)
plt.tight_layout()
plt.show()

print(f"Correct: {np.trace(cm):,} / {cm.sum():,}   Wrong: {cm.sum() - np.trace(cm):,}")

In [ ]:
per_class = pd.DataFrame({
    "digit": range(10),
    "support": cm.sum(axis=1),
    "correct": np.diag(cm),
    "accuracy": np.diag(cm) / cm.sum(axis=1),
})
per_class["errors"] = per_class["support"] - per_class["correct"]

# the most frequently confused ordered pairs
off = cm.copy()
np.fill_diagonal(off, 0)
pairs = [(off[i, j], i, j) for i in range(10) for j in range(10) if off[i, j] > 0]
pairs.sort(reverse=True)
confused = pd.DataFrame([{"true": i, "predicted": j, "count": c,
                          "% of true class": round(100 * c / cm[i].sum(), 2)}
                         for c, i, j in pairs[:8]])

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8),
                         gridspec_kw={"width_ratios": [1.5, 1]})

ax = axes[0]
bars = ax.bar(per_class["digit"], 100 * per_class["accuracy"], color=PALETTE[0])
worst = per_class["accuracy"].idxmin()
bars[worst].set_color(PALETTE[1])
for b, v, e in zip(bars, 100 * per_class["accuracy"], per_class["errors"]):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%\n({e} err)",
            ha="center", va="bottom", fontsize=7)
ax.set_xticks(range(10))
ax.set_ylim(max(90, 100 * per_class["accuracy"].min() - 2), 100.8)
ax.set_xlabel("digit")
ax.set_ylabel("recall (%)")
ax.set_title(f"(a) Per-class accuracy - hardest digit is {worst} "
             f"(orange)")

ax = axes[1]
ax.axis("off")
ax.set_title("(b) Most confused pairs", loc="left")
ax.table(cellText=confused.values, colLabels=confused.columns,
         loc="center", cellLoc="center").scale(1, 1.25)

plt.tight_layout()
plt.show()

print("Most confused pairs:")
print(confused.to_string(index=False))
print()

In [ ]:
wrong = np.where(y_pred != y_test)[0]
# show the mistakes the model was most CONFIDENT about - the genuinely interesting failures
wrong_sorted = wrong[np.argsort(-y_conf[wrong])]

n_show = min(18, len(wrong_sorted))
print(f"{len(wrong)} misclassified out of {len(y_test)} "
      f"({100 * len(wrong) / len(y_test):.2f}%). "
      f"Showing the {n_show} the model was most confident about.\n")

fig, axes = plt.subplots(3, 6, figsize=(12, 6.4))
for ax, idx in zip(axes.ravel(), wrong_sorted[:n_show]):
    ax.imshow(x_test[idx].squeeze(), cmap="gray")
    ax.set_title(f"true {y_test[idx]}  ->  pred {y_pred[idx]}\nconfidence {y_conf[idx]:.0%}",
                 fontsize=8, color="#c92a2a")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
for ax in axes.ravel()[n_show:]:
    ax.axis("off")
fig.suptitle("The remaining mistakes, ordered by the model's confidence", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
correct_mask = y_pred == y_test

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# (a) how confident is the model, and is confidence informative? ------------------------------
ax = axes[0]
bins = np.linspace(0, 1, 41)
ax.hist(y_conf[correct_mask], bins=bins, color=PALETTE[2], label="correct predictions")
ax.hist(y_conf[~correct_mask], bins=bins, color=PALETTE[1], label="wrong predictions")
ax.set_yscale("log")
ax.set_xlabel("confidence (highest softmax probability)")
ax.set_ylabel("number of test images (log scale)")
ax.set_title("(a) Confidence separates right from wrong")
ax.legend()
print(f"Mean confidence when correct : {y_conf[correct_mask].mean():.4f}")
print(f"Mean confidence when wrong   : {y_conf[~correct_mask].mean():.4f}")

# (b) would a "refer to a human" threshold make this deployable? ------------------------------
ax = axes[1]
thresholds = np.linspace(0.5, 0.999, 60)
coverage, accuracy = [], []
for t in thresholds:
    keep = y_conf >= t
    coverage.append(keep.mean())
    accuracy.append(correct_mask[keep].mean() if keep.any() else np.nan)
ax.plot(100 * np.array(coverage), 100 * np.array(accuracy), color=PALETTE[0])
ax.set_xlabel("coverage: % of images the model answers automatically")
ax.set_ylabel("accuracy on the images it answers (%)")
ax.set_title("(b) Accuracy vs coverage if low-confidence cases\nare referred to a human")
ax.invert_xaxis()
for t in (0.9, 0.99, 0.999):
    keep = y_conf >= t
    if keep.any():
        ax.scatter(100 * keep.mean(), 100 * correct_mask[keep].mean(),
                   color=PALETTE[1], zorder=3)
        ax.annotate(f"t={t}", (100 * keep.mean(), 100 * correct_mask[keep].mean()),
                    textcoords="offset points", xytext=(6, -10), fontsize=8)
plt.tight_layout()
plt.show()

print()
for t in (0.90, 0.99, 0.999):
    keep = y_conf >= t
    if keep.any():
        print(f"  threshold {t:<6} -> answers {100 * keep.mean():5.1f}% of images "
              f"at {100 * correct_mask[keep].mean():.3f}% accuracy, "
              f"refers {(~keep).sum():4d} images to a human")

## 12. What the CNN Learned

In [ ]:
# use the best single network (an ensemble has no single set of filters to show)
best_single = max(models, key=lambda n: next(r["val_acc"] for r in results
                                             if r["experiment"] == n))
net = models[best_single]
print(f"Inspecting: {best_single}  ({net.name})")

conv_layers = [l for l in net.layers if isinstance(l, layers.Conv2D)]
W = conv_layers[0].get_weights()[0]              # (3, 3, 1, n_filters)
n_filt = W.shape[-1]
vmax = np.abs(W).max()

cols = 8
rows = int(np.ceil(n_filt / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.15, rows * 1.3))
for i, ax in enumerate(axes.ravel()):
    if i < n_filt:
        ax.imshow(W[:, :, 0, i], cmap="coolwarm", vmin=-vmax, vmax=vmax)
        ax.set_title(f"#{i}", fontsize=7)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    if i >= n_filt:
        ax.axis("off")
fig.suptitle(f"The {n_filt} learned 3x3 filters of the first convolutional layer\n"
             f"(blue = negative weight, red = positive) - compare with the hand-crafted "
             f"kernel in section 4  [slides p. 16-17]", fontsize=9.5)
plt.tight_layout()
plt.show()

In [ ]:
# Push one digit through the network and show the feature maps after each stage [slides p. 19-22]
probe_idx = int(np.where(y_test == 8)[0][0])
z = x_test[probe_idx][np.newaxis, ...]

stages = []
for layer in net.layers:
    z = layer(z, training=False)
    if isinstance(layer, (layers.Conv2D, layers.MaxPooling2D)):
        stages.append((layer.__class__.__name__, tuple(z.shape[1:]), np.array(z)[0]))

n_show = 8
fig, axes = plt.subplots(len(stages) + 1, n_show, figsize=(n_show * 1.2, (len(stages) + 1) * 1.35))

axes[0, 0].imshow(x_test[probe_idx].squeeze(), cmap="gray")
axes[0, 0].set_ylabel(f"input\n28x28x1", fontsize=7.5, rotation=0, ha="right", va="center")
for ax in axes[0, 1:]:
    ax.axis("off")

for row, (kind, shape, maps) in enumerate(stages, start=1):
    order = np.argsort(-maps.reshape(-1, maps.shape[-1]).max(axis=0))  # liveliest maps first
    for col in range(n_show):
        ax = axes[row, col]
        if col < maps.shape[-1]:
            ax.imshow(maps[:, :, order[col]], cmap="viridis")
        else:
            ax.axis("off")
    axes[row, 0].set_ylabel(f"{kind}\n{shape[0]}x{shape[1]}x{shape[2]}",
                            fontsize=7.5, rotation=0, ha="right", va="center")

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
fig.suptitle(f"Feature maps for one test image of a {y_test[probe_idx]} "
             f"(8 strongest maps per stage)\nEarly layers respond to strokes and edges; after "
             f"pooling the maps are smaller and more abstract  [slides p. 19-22]", fontsize=9.5)
plt.tight_layout()
plt.show()

print(f"After the last pooling stage the image is {stages[-1][1]} = "
      f"{int(np.prod(stages[-1][1]))} numbers, which is exactly what Flatten turns into a "
      f"vector for\nthe fully connected layers [slides p. 22]. The network has gone from "
      f"784 raw pixels to a compact,\nlearned description of the digit's shape.")

## 13. 5-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=CFG["cv_folds"], shuffle=True, random_state=SEED)
fold_rows = []

print(f"Stratified {CFG['cv_folds']}-fold cross-validation of cnn_bn "
      f"({CFG['cv_epochs']} epochs per fold)\n")
cv_t0 = time.perf_counter()

for fold, (tr_idx, va_idx) in enumerate(skf.split(x_train_full, y_train_full), start=1):
    keras.utils.set_random_seed(SEED + fold)
    model = cnn_bn()
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(x_train_full[tr_idx], y_train_full[tr_idx],
              epochs=CFG["cv_epochs"], batch_size=128, verbose=0)
    acc = model.evaluate(x_train_full[va_idx], y_train_full[va_idx], verbose=0)[1]
    fold_rows.append({"fold": fold, "train_size": len(tr_idx),
                      "val_size": len(va_idx), "accuracy": round(float(acc), 4)})
    print(f"  fold {fold}/{CFG['cv_folds']}:  "
          f"train {len(tr_idx):>6,}  held-out {len(va_idx):>6,}  accuracy {acc:.4f}")
    del model
    keras.backend.clear_session()

keras.utils.set_random_seed(SEED)
cv_df = pd.DataFrame(fold_rows)
cv_mean, cv_std = cv_df["accuracy"].mean(), cv_df["accuracy"].std(ddof=1)

print(f"\n  mean accuracy : {cv_mean:.4f}")
print(f"  std deviation : {cv_std:.4f}")
print(f"  spread        : {cv_df['accuracy'].min():.4f} to {cv_df['accuracy'].max():.4f}")
print(f"  total time    : {time.perf_counter() - cv_t0:.0f}s")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
bars = ax.bar(cv_df["fold"], cv_df["accuracy"], color=PALETTE[0], width=0.6)
ax.axhline(cv_mean, color=PALETTE[1], ls="--", label=f"mean {cv_mean:.4f}")
ax.fill_between([0.4, CFG["cv_folds"] + 0.6], cv_mean - cv_std, cv_mean + cv_std,
                color=PALETTE[1], alpha=0.12, label=f"+/- 1 std ({cv_std:.4f})")
for b, v in zip(bars, cv_df["accuracy"]):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.4f}", ha="center", va="bottom", fontsize=8)
ax.set_xlim(0.4, CFG["cv_folds"] + 0.6)
ax.set_ylim(cv_df["accuracy"].min() - 0.008, min(1.0, cv_df["accuracy"].max() + 0.006))
ax.set_xticks(cv_df["fold"])
ax.set_xlabel("fold")
ax.set_ylabel("held-out accuracy")
ax.set_title(f"Stratified {CFG['cv_folds']}-fold cross-validation of cnn_bn")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 14. Save, Reload, Predict

In [ ]:
MODEL_PATH = "best_cnn_model.keras"

net.save(MODEL_PATH)                                      # same step as model.save [p. 26]
reloaded = keras.models.load_model(MODEL_PATH)            # same step as load_model [p. 27]

orig_acc = net.evaluate(x_test, y_test, verbose=0)[1]
load_acc = reloaded.evaluate(x_test, y_test, verbose=0)[1]
print(f"Saved      : {MODEL_PATH}  ({best_single})")
print(f"Accuracy before saving : {orig_acc:.4f}")
print(f"Accuracy after loading : {load_acc:.4f}   "
      f"{'identical - the saved file is complete' if abs(orig_acc - load_acc) < 1e-6 else 'MISMATCH'}")

# predict one single image, exactly as the slides do [p. 27]
i = 30
pred = reloaded.predict(x_test[i: i + 1], verbose=0)[0]
print(f"\nSingle-image prediction (test image #{i})")
print(f"  label actual     : {y_test[i]}")
print(f"  label prediction : {np.argmax(pred)}   (confidence {pred.max():.2%})")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), gridspec_kw={"width_ratios": [1, 2]})
axes[0].imshow(x_test[i].squeeze(), cmap="gray")
axes[0].set_title(f"test image #{i}\ntrue label: {y_test[i]}")
axes[0].set_xticks([])
axes[0].set_yticks([])
axes[0].grid(False)

colors = [PALETTE[2] if d == np.argmax(pred) else PALETTE[5] for d in range(10)]
axes[1].bar(range(10), pred, color=colors)
axes[1].set_xticks(range(10))
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("digit")
axes[1].set_ylabel("softmax probability")
axes[1].set_title("The 10 output probabilities (they sum to 1)")
for d in range(10):
    if pred[d] > 0.01:
        axes[1].text(d, pred[d], f"{pred[d]:.2f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

## 15. Conclusion

In [ ]:
b = summary.loc[0]
r = {row["experiment"]: row for row in results}
print(f"Best model : {b['experiment']}  val {b['val_acc']:.4f}  test {b['test_acc']:.4f}  "
      f"(error {b['test_err_%']:.2f}%, {int(round((1 - b['test_acc']) * len(y_test)))} of {len(y_test):,})")
print(f"5-fold CV  : {cv_mean:.4f} +/- {cv_std:.4f}")
print()
prev = None
for key in ["E1_ann", "E2_cnn_slide", "E3_cnn_wide", "E4_cnn_bn", "E5_cnn_bn_aug", "E6_ensemble"]:
    err = r[key]["test_err_%"]
    print(f"{key:<15} test error {err:>5.2f}%" + ("" if prev is None else f"   ({err - prev:+.2f} pp)"))
    prev = err
print(f"\nMean confidence: {y_conf[correct_mask].mean():.3f} when correct, "
      f"{y_conf[~correct_mask].mean():.3f} when wrong")

## 16. Reference

Afiahayati. *Introduction to Deep Learning* (course slides). Universitas Gadjah Mada.